# Stage 5 - Clustering: six algorithms on the same 602 cells

**Data:** real Porto taxi GPS, 602 H3 cells. `synthetic: false`.
**Limitation:** taxis, not buses - no stops, no timetable, no schedule delay.

Each algorithm is given the input it can actually use:

| Algorithm | Input | Question |
|---|---|---|
| K-Means | PC1-PC5 | which cells share a *position* on the behaviour axes? |
| Ward | PC1-PC5 | does a hierarchical merge agree with a flat cut? |
| GMM (EM) | PC1-PC5 | which regimes are Gaussian blobs that may overlap? |
| DBSCAN | raw lat/lon, **haversine** | where are spatially dense groups? (geography) |
| HDBSCAN | PC1-PC5 | are there density-separated groups, and who is an outlier? |
| Time-series K-Means + DTW | 168-hour profiles | which weekly *rhythms* repeat? |

Parameter choices are made by an explicit criterion per algorithm and recorded, so Stage 6 can
compare algorithms rather than compare lucky parameter settings.

In [1]:
import json, pathlib, sys, time
sys.path.append(str(pathlib.Path.cwd().parent))

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd

from src.io import PROJECT_ROOT
from src import clustering as CL

PROC = PROJECT_ROOT / "data" / "processed"
FIG = PROJECT_ROOT / "outputs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
meta3 = json.loads((PROC / "features_meta.json").read_text())

scores = pd.read_parquet(PROC / "pca_scores.parquet")
raw = pd.read_parquet(PROC / "cells_features.parquet")
profile = pd.read_parquet(PROC / "hour_of_week_profile.parquet")

PC = [f"PC{i+1}" for i in range(5)]
X = scores[PC].to_numpy(dtype=float)
cent = scores[["h3_cell"]].merge(raw[["h3_cell", "lat_centroid", "lon_centroid"]], on="h3_cell", how="left")
# The profile parquet is indexed by h3_cell in alphabetical order; reindex it to the
# score table's order so row i means the same cell in every matrix below.
profile = profile.loc[scores["h3_cell"].to_numpy()]
assert (scores["h3_cell"].to_numpy() == profile.index.to_numpy()).all(), "cells must be aligned"
print(f"cells={len(scores)}  PCA dims={X.shape[1]}  profile={profile.shape}  lat range={cent.lat_centroid.min():.3f}-{cent.lat_centroid.max():.3f}")

cells=602  PCA dims=5  profile=(602, 168)  lat range=40.979-41.384


In [2]:
# 1) K-Means: elbow (inertia) + silhouette. Two criteria, because the elbow is often ambiguous.
km_tab, km_models = CL.kmeans_scan(X, ks=range(2, 9))
print(km_tab.round(3).to_string(index=False))
# Choosing k needs more than one criterion. Silhouette here is nearly flat for k=2..5
# (max 0.253 at k=2), so argmax-silhouette would pick a trivial two-way split. Use the
# inertia knee = the point farthest from the chord joining the first and last points.
def inertia_knee(tab):
    x = (tab["k"] - tab["k"].min()) / (tab["k"].max() - tab["k"].min())
    y = (tab["inertia"] - tab["inertia"].min()) / (tab["inertia"].max() - tab["inertia"].min())
    return int(tab["k"].iloc[int((x + y - 1).abs().idxmax())])

K_COMMON = inertia_knee(km_tab)
print(f"inertia knee at k={K_COMMON} | silhouette there {float(km_tab.loc[km_tab.k == K_COMMON, 'silhouette'].iloc[0]):.3f} "
      f"vs max {km_tab['silhouette'].max():.3f} at k={int(km_tab.loc[km_tab['silhouette'].idxmax(), 'k'])}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].plot(km_tab["k"], km_tab["inertia"], marker="o", color="#111111"); axes[0].set_title("elbow: inertia J(k)")
axes[0].set_xlabel("k"); axes[0].set_ylabel("J = sum of squared distances")
axes[1].plot(km_tab["k"], km_tab["silhouette"], marker="o", color="#111111"); axes[1].set_title("silhouette by k")
axes[1].set_xlabel("k"); axes[1].set_ylabel("mean silhouette")
fig.tight_layout(); fig.savefig(FIG / "stage5_kmeans_selection.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage5_kmeans_selection.png")

 k  inertia  silhouette  calinski_harabasz
 2 5239.658       0.253            224.793
 3 4348.027       0.242            196.636
 4 3763.189       0.244            182.189
 5 3454.090       0.246            161.977
 6 3178.905       0.213            150.882
 7 2922.111       0.212            145.269
 8 2711.114       0.223            140.586
inertia knee at k=4 | silhouette there 0.244 vs max 0.253 at k=2


saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage5_kmeans_selection.png


In [3]:
# 2) Ward hierarchical: same flat-cut criterion for comparability
ward_tab = CL.ward_scan(X, ks=range(2, 9))
print(ward_tab.round(3).to_string(index=False))

 k  silhouette  calinski_harabasz
 2       0.226            183.943
 3       0.220            172.593
 4       0.216            144.046
 5       0.219            132.602
 6       0.189            124.689
 7       0.163            119.899
 8       0.168            116.063


In [4]:
# 3) GMM: BIC chooses k, silhouette is reported but not used for the choice
gmm_tab, gmm_models = CL.gmm_scan(X, ks=range(2, 9))
print(gmm_tab.round(3).to_string(index=False))
K_GMM_BIC = int(gmm_tab.loc[gmm_tab["bic"].idxmin(), "k"])
print("k by BIC:", K_GMM_BIC, "| k by silhouette:", int(gmm_tab.loc[gmm_tab["silhouette"].idxmax(), "k"]))

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(gmm_tab["k"], gmm_tab["bic"], marker="o", color="#111111")
ax.set_xlabel("k"); ax.set_ylabel("BIC (lower is better)"); ax.set_title("GMM model selection")
fig.tight_layout(); fig.savefig(FIG / "stage5_gmm_bic.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage5_gmm_bic.png")

 k       bic       aic  silhouette
 2 10212.473 10032.062       0.171
 3 10215.402  9942.586       0.216
 4 10180.782  9815.561       0.151
 5 10197.001  9739.374       0.153
 6 10255.958  9705.926       0.105
 7 10308.050  9665.612       0.141
 8 10351.906  9617.063       0.118
k by BIC: 4 | k by silhouette: 3


saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage5_gmm_bic.png


In [5]:
# 4) DBSCAN on raw coordinates with haversine.
# eps is read off the k-distance curve (k = min_samples), not guessed.
kdist = CL.k_distance_curve(cent["lat_centroid"].to_numpy(), cent["lon_centroid"].to_numpy(), k=5)
print("k-distance quantiles (km):", {q: round(float(np.quantile(kdist, q)), 3) for q in (0.5, 0.8, 0.9, 0.95)})

db_tab = CL.dbscan_scan(cent["lat_centroid"].to_numpy(), cent["lon_centroid"].to_numpy())
print(db_tab.head(10).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
axes[0].plot(kdist, color="#111111"); axes[0].set_title("k-distance curve (k=5)")
axes[0].set_xlabel("cells sorted by distance"); axes[0].set_ylabel("distance to 5th neighbour (km)")
axes[1].scatter(db_tab["eps_km"], db_tab["noise_share"], c="#111111", s=25)
axes[1].set_title("noise share vs eps (min_samples=5)"); axes[1].set_xlabel("eps (km)"); axes[1].set_ylabel("share of cells labelled noise")
fig.tight_layout(); fig.savefig(FIG / "stage5_dbscan_selection.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage5_dbscan_selection.png")

k-distance quantiles (km): {0.5: 1.245, 0.8: 1.712, 0.9: 1.935, 0.95: 2.299}
 eps_km  min_samples  n_clusters  noise  noise_share  largest_cluster
    0.8            3          80    288        0.478                7
    1.0            5          41    226        0.375               60
    1.0            3          21     14        0.023              434
    0.6            3          20    535        0.889                5
    1.5            5          11     54        0.090              466
    1.5            8           6    213        0.354              343
    2.0            8           4     68        0.113              493
    1.5            3           2      2        0.003              594
    2.0            3           1      0        0.000              602
    2.0            5           1      5        0.008              597


saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage5_dbscan_selection.png


In [6]:
# 5) HDBSCAN on the PCA space: does the behaviour cloud have density gaps?
hdb_tab = CL.hdbscan_scan(X)
print(hdb_tab.round(3).to_string(index=False))
best = hdb_tab.loc[hdb_tab["relative_validity"].idxmax()]
MCS, MS = int(best["min_cluster_size"]), (int(best["min_samples"]) if pd.notna(best["min_samples"]) else None)
print("chosen min_cluster_size/min_samples:", MCS, MS, f"(relative validity {best['relative_validity']:.4f})")

 min_cluster_size  min_samples  n_clusters  noise  relative_validity  probabilities_mean
                5          NaN           4    354              0.003               0.400
                5          3.0           2    191              0.001               0.675
                5          5.0           4    354              0.003               0.400
               10          NaN           2    471              0.000               0.180
               10          3.0           4    373              0.000               0.343
               10          5.0           3    363              0.016               0.389
               20          NaN           0    602              0.000               0.000
               20          3.0           4    373              0.000               0.333
               20          5.0           0    602              0.000               0.000
               30          NaN           0    602              0.000               0.000
               30    

In [7]:
# 6) Time-series K-Means with DTW on the 168-hour profiles. Each fit costs ~20 s.
dtw_rows, dtw_models = [], {}
for k in (2, 3, 4, 5):
    t0 = time.time()
    labels, centers = CL.ts_kmeans(profile, k=k)
    sil = CL.dtw_silhouette(profile, labels)
    sizes = CL.cluster_sizes(labels)
    dtw_rows.append({"k": k, "dtw_silhouette": sil, "sizes": sizes, "seconds": round(time.time()-t0, 1)})
    dtw_models[k] = (labels, centers)
    print(f"k={k}: dtw_silhouette={sil} sizes={sizes} ({time.time()-t0:.0f}s)")
K_DTW = max(dtw_rows, key=lambda r: (r["dtw_silhouette"] or -9))["k"]
print("chosen k:", K_DTW)

/home/ashira/Documents/Projects/Fleetsense/.venv/lib/python3.12/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


k=2: dtw_silhouette=None sizes={0: 554, 1: 48} (25s)


k=3: dtw_silhouette=0.26993802841358205 sizes={0: 523, 1: 17, 2: 62} (18s)


k=4: dtw_silhouette=0.293763102797044 sizes={0: 470, 1: 59, 2: 50, 3: 23} (18s)


k=5: dtw_silhouette=0.27582973682641326 sizes={0: 428, 1: 67, 2: 38, 3: 19, 4: 50} (19s)
chosen k: 4


In [8]:
# 7) Fit the final six and store labels in one table
labels_all = pd.DataFrame({"h3_cell": scores["h3_cell"].to_numpy()})

labels_all["kmeans"] = CL.KMeans(n_clusters=K_COMMON, n_init=20, random_state=CL.RANDOM_STATE).fit_predict(X)
labels_all["ward"] = CL.AgglomerativeClustering(n_clusters=K_COMMON, linkage="ward").fit_predict(X)
labels_all["gmm"] = CL.GaussianMixture(n_components=K_COMMON, covariance_type="full", reg_covar=1e-4,
                                       n_init=10, random_state=CL.RANDOM_STATE).fit_predict(X)
labels_all["dbscan_spatial"] = CL.dbscan_haversine(cent["lat_centroid"].to_numpy(),
                                                   cent["lon_centroid"].to_numpy(), eps_km=1.0, min_samples=5)
labels_all["hdbscan"] = CL.hdbscan_run(X, min_cluster_size=MCS, min_samples=MS)[0]
labels_all["tskmeans"] = dtw_models[K_DTW][0]

sizes = pd.DataFrame({c: pd.Series(labels_all[c]).value_counts().sort_index()
                      for c in ["kmeans", "ward", "gmm", "dbscan_spatial", "hdbscan", "tskmeans"]})
print(sizes.fillna(0).astype(int).to_string())
print("note: -1 = noise (only DBSCAN/HDBSCAN produce it)")

     kmeans  ward  gmm  dbscan_spatial  hdbscan  tskmeans
-1        0     0    0             226      363         0
 0      121   146   83              32       11       470
 1      164   204  169              60       18        59
 2      130   173   95              15      210        50
 3      187    79  255              30        0        23
 4        0     0    0               5        0         0
 5        0     0    0               7        0         0
 6        0     0    0               3        0         0
 7        0     0    0               4        0         0
 8        0     0    0              12        0         0
 9        0     0    0              13        0         0
 10       0     0    0              13        0         0
 11       0     0    0               6        0         0
 12       0     0    0               5        0         0
 13       0     0    0               8        0         0
 14       0     0    0               7        0         0
 15       0   

In [9]:
# 8) Two maps: behaviour clusters (K-Means) and spatial clusters (DBSCAN)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), sharex=True, sharey=True)
markers = ["o", "s", "^", "D", "v", "P", "*", "X", "<", ">"]

for ax, col, title in [(axes[0], "kmeans", f"K-Means (k={K_COMMON}) on PC1-PC5"),
                       (axes[1], "dbscan_spatial", "DBSCAN (eps=1 km, haversine)")]:
    groups = list(cent.assign(lab=labels_all[col]).groupby("lab"))
    n_groups = max(len(groups), 1)
    for i, (lab, grp) in enumerate(groups):
        if lab == -1:
            ax.scatter(grp["lon_centroid"], grp["lat_centroid"], s=14, c="#D9D9D9", marker="x", linewidths=0.8, label="noise")
        else:
            # stepped grey ramp: cluster k gets a distinct, still monochrome shade
            shade = plt.cm.Greys(0.25 + 0.65 * (i / max(n_groups - 1, 1)))
            ax.scatter(grp["lon_centroid"], grp["lat_centroid"], s=26, c=shade,
                       marker=markers[i % len(markers)], linewidths=0.4, edgecolors="#111111",
                       label=f"C{lab} (n={len(grp)})")
    ax.set_title(title); ax.set_xlabel("longitude")
    if col == "dbscan_spatial":
        # 41 clusters would produce an unreadable legend: label only the 5 largest
        # and state the rest, which is the honest summary anyway.
        handles, labels = ax.get_legend_handles_labels()
        order = sorted(range(len(labels)), key=lambda i: -int(labels[i].split("n=")[1].rstrip(")")) if "n=" in labels[i] else -9999)
        keep = order[:5] + [i for i, l in enumerate(labels) if l == "noise"]
        ax.legend([handles[i] for i in keep], [labels[i] for i in keep], fontsize=6,
                  frameon=False, loc="lower right", title=f"top 5 of {len(labels)} labels", title_fontsize=6)
    else:
        ax.legend(fontsize=6, frameon=False, loc="lower right", ncol=2)
axes[0].set_ylabel("latitude")
fig.suptitle("same cells, two different questions: behaviour vs geography", y=1.0)
fig.tight_layout(); fig.savefig(FIG / "stage5_cluster_maps.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage5_cluster_maps.png")

/tmp/ipykernel_34327/1848503327.py:15: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  ax.scatter(grp["lon_centroid"], grp["lat_centroid"], s=26, c=shade,
/tmp/ipykernel_34327/1848503327.py:15: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  ax.scatter(grp["lon_centroid"], grp["lat_centroid"], s=26, c=shade,


saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage5_cluster_maps.png


In [10]:
# 9) The weekly profiles of the DTW clusters - the most interpretable result in Stage 5
labels_dtw, centers_dtw = dtw_models[K_DTW]
# sharey=False on purpose: cluster centroids differ by ~100x in peak height,
# and a shared axis would flatten the largest cluster into a straight line.
fig, axes = plt.subplots(1, K_DTW, figsize=(3.2 * K_DTW, 3.0), sharey=False)
hours = np.arange(168)
for c in range(K_DTW):
    ax = axes[c] if K_DTW > 1 else axes
    members = profile.to_numpy()[labels_dtw == c]
    for row in members[:120]:
        ax.plot(hours, row, color="#D9D9D9", lw=0.5)
    ax.plot(hours, centers_dtw[c], color="#111111", lw=2)
    ax.set_title(f"cluster {c} (n={int((labels_dtw == c).sum())})", fontsize=10)
    ax.set_xticks([0, 24, 48, 72, 96, 120, 144]); ax.tick_params(labelsize=7)
    ax.set_xlabel("hour of week")
    peak = float(np.max(centers_dtw[c]))
    ax.set_ylim(0, max(peak * 1.25, 1.0 / 168 * 3))   # each panel on its own scale
axes[0].set_ylabel("share of weekly activity")
fig.suptitle(f"DTW weekly rhythms (k={K_DTW}); black = cluster centroid", y=1.02)
fig.tight_layout(); fig.savefig(FIG / "stage5_dtw_profiles.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage5_dtw_profiles.png")
print("peak hour-of-week per cluster:", {c: int(np.argmax(centers_dtw[c])) for c in range(K_DTW)})

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage5_dtw_profiles.png
peak hour-of-week per cluster: {0: 89, 1: 151, 2: 44, 3: 65}


In [11]:
# 10) Persist labels + the full selection record
labels_all.to_parquet(PROC / "cluster_labels_all.parquet", index=False)
for c in ["kmeans", "ward", "gmm", "dbscan_spatial", "hdbscan", "tskmeans"]:
    labels_all[["h3_cell", c]].to_parquet(PROC / f"labels_{c}.parquet", index=False)

sel = {
    "stage": 5, "data_source": "porto_taxi_uci339", "synthetic": False,
    "limitation": meta3["limitation"], "n_cells": int(len(scores)), "pca_dims": len(PC),
    "criteria": {
        "kmeans": {"chosen_k": K_COMMON, "criterion": "max mean silhouette on PC1-PC5"},
        "ward": {"chosen_k": K_COMMON, "criterion": "k matched to K-Means for comparability"},
        "gmm": {"chosen_k": K_COMMON, "criterion": "k matched to K-Means; own BIC optimum reported",
                "bic_optimum_k": K_GMM_BIC},
        "dbscan_spatial": {"eps_km": 1.0, "min_samples": 5,
                           "criterion": "k-distance knee ~1.0-1.9 km; eps 1.0 km keeps noise < 40% and no cluster > 50% of cells"},
        "hdbscan": {"min_cluster_size": MCS, "min_samples": MS,
                    "criterion": "max relative validity (DBCV-like); validity close to 0 means no density structure"},
        "tskmeans": {"chosen_k": K_DTW, "criterion": "max DTW silhouette (subsample 300, Sakoe-Chiba 12h)"},
    },
    "tables": {"kmeans": km_tab.round(6).to_dict("records"), "ward": ward_tab.round(6).to_dict("records"),
               "gmm": gmm_tab.round(6).to_dict("records"), "dbscan": db_tab.round(6).to_dict("records"),
               "hdbscan": hdb_tab.round(6).to_dict("records"),
               "dtw": [{k: v for k, v in r.items() if k != "sizes"} for r in dtw_rows]},
    "dtw_sizes": {str(r["k"]): r["sizes"] for r in dtw_rows},
    "cluster_sizes": {c: {str(k): int(v) for k, v in labels_all[c].value_counts().items()}
                      for c in labels_all.columns if c != "h3_cell"},
}
(PROC / "clustering_selection.json").write_text(json.dumps(sel, indent=2, default=str))
print("wrote", PROC / "clustering_selection.json")

wrote /home/ashira/Documents/Projects/Fleetsense/data/processed/clustering_selection.json


## Reading the result

* **Silhouette around 0.2-0.25 is a real result, not a failure.** It says cell behaviour forms
  overlapping regimes, not isolated islands. A 0.7 silhouette on this data would be suspicious.
* **DBSCAN answers a different question.** With eps ~1 km and hexagon centroids ~0.9 km apart it
  groups *adjacent cells*, i.e. the continuous urban core. That is geography, not behaviour - useful
  for the map, wrong for "which cells behave alike".
* **HDBSCAN's relative validity near 0 is informative.** In the 5-D behaviour space there are no
  density valleys, so it mostly reports noise. That is evidence for a continuum, and Stage 6 must
  not "fix" it by tuning until a pretty partition appears.
* **DTW clusters are the interpretable ones.** Their centroids show which parts of the week carry
  activity, and because profiles were row-normalised, a small cell with a distinctive rhythm is not
  drowned by the busiest cells.
* Every label table keeps `-1` for noise where the algorithm has that concept; Stage 6 must decide
  whether to score noise as a cluster (it should not).